# A Fool Fraud — Data Augmentation
**Improving Fraud Detection with SMOTE, ADASYN, and GANs**

This notebook applies state-of-the-art data augmentation techniques to tackle class imbalance,
then compares the augmented models against the traditional baseline from `a_fool_fraud_tradi.ipynb`.

### Techniques:
| Technique | Description |
|---|---|
| **SMOTE** | Synthetic Minority Over-sampling Technique — interpolates between minority neighbors |
| **ADASYN** | Adaptive Synthetic — focuses oversampling on harder-to-learn samples |
| **GAN** | Generative Adversarial Network — learns the true fraud data distribution |

### Models re-evaluated after augmentation:
- Logistic Regression, Random Forest, SVM, Gradient Boosting

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & Models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

# Resampling — requires: pip install imbalanced-learn
from imblearn.over_sampling import SMOTE, ADASYN

# Deep learning for GAN — requires: pip install tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('All libraries loaded successfully.')

## 2. Load & Preprocess Data

In [ ]:
df = pd.read_csv('creditcard.csv')

# Scale Amount and Time
scaler = StandardScaler()
df['scaled_Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_Time']   = scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df_clean = df.drop(['Amount', 'Time'], axis=1)

X = df_clean.drop('Class', axis=1)
y = df_clean['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}  |  Fraud in train: {y_train.sum():,} ({y_train.mean()*100:.3f}%)')
print(f'Test : {X_test.shape}   |  Fraud in test : {y_test.sum():,} ({y_test.mean()*100:.3f}%)')

## 3. Helper: Train & Evaluate All Models

In [ ]:
def get_models():
    return {
        'Logistic Regression': LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
        ),
        'SVM': SVC(
            kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=100, random_state=RANDOM_STATE
        )
    }


def evaluate_models(X_tr, y_tr, X_te, y_te, label=''):
    """Train all 4 classifiers and return metrics dict."""
    results = {}
    for name, model in get_models().items():
        model.fit(X_tr, y_tr)
        y_pred      = model.predict(X_te)
        y_pred_prob = model.predict_proba(X_te)[:, 1]
        results[name] = {
            'model'      : model,
            'y_pred'     : y_pred,
            'y_pred_prob': y_pred_prob,
            'precision'  : precision_score(y_te, y_pred),
            'recall'     : recall_score(y_te, y_pred),
            'f1'         : f1_score(y_te, y_pred),
            'roc_auc'    : roc_auc_score(y_te, y_pred_prob),
            'avg_prec'   : average_precision_score(y_te, y_pred_prob),
        }
        print(f'  [{label}] {name:22s}  F1={results[name]["f1"]:.4f}  ROC-AUC={results[name]["roc_auc"]:.4f}')
    return results


def summary_table(results_dict):
    """Build a summary DataFrame from results."""
    return pd.DataFrame({
        name: {
            'Precision'    : round(v['precision'], 4),
            'Recall'       : round(v['recall'], 4),
            'F1 Score'     : round(v['f1'], 4),
            'ROC-AUC'      : round(v['roc_auc'], 4),
            'Avg Precision': round(v['avg_prec'], 4),
        }
        for name, v in results_dict.items()
    }).T.sort_values('F1 Score', ascending=False)


print('Helper functions defined.')

## 4. Baseline Results (No Augmentation)
We replicate traditional model performance here for direct comparison.

In [ ]:
print('Training baseline models (no augmentation)...')
baseline_results = evaluate_models(X_train, y_train, X_test, y_test, label='Baseline')

print('\nBaseline Summary:')
display(summary_table(baseline_results).style.highlight_max(color='lightgreen'))

## 5. SMOTE — Synthetic Minority Over-sampling Technique

SMOTE works by selecting a minority sample, finding its k-nearest minority neighbors,
and creating synthetic points along the line segments between them.

> ⚠️ **Important:** SMOTE is only applied to the **training set**, never the test set.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE → Fraud: {y_train.sum():,} / Total: {len(y_train):,} ({y_train.mean()*100:.3f}%)')
print(f'After  SMOTE → Fraud: {y_train_smote.sum():,} / Total: {len(y_train_smote):,} ({y_train_smote.mean()*100:.3f}%)')

# Visualize the effect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, title in zip(
    axes,
    [y_train.value_counts(), pd.Series(y_train_smote).value_counts()],
    ['Before SMOTE', 'After SMOTE']
):
    ax.bar(['Genuine', 'Fraud'], counts.values, color=['#2196F3', '#F44336'], edgecolor='black')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

plt.suptitle('Class Balance: Before vs After SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('smote_balance.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Training models on SMOTE-augmented data...')
smote_results = evaluate_models(X_train_smote, y_train_smote, X_test, y_test, label='SMOTE')

print('\nSMOTE Summary:')
display(summary_table(smote_results).style.highlight_max(color='lightgreen'))

## 6. ADASYN — Adaptive Synthetic Sampling

ADASYN is similar to SMOTE but adapts the density of synthetic samples:
it generates **more synthetic samples** for minority class samples that are harder to learn
(those surrounded by more majority class samples).

In [ ]:
adasyn = ADASYN(random_state=RANDOM_STATE, n_neighbors=5)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print(f'Before ADASYN → Fraud: {y_train.sum():,} / Total: {len(y_train):,}')
print(f'After  ADASYN → Fraud: {y_train_adasyn.sum():,} / Total: {len(y_train_adasyn):,}')

print('\nTraining models on ADASYN-augmented data...')
adasyn_results = evaluate_models(X_train_adasyn, y_train_adasyn, X_test, y_test, label='ADASYN')

print('\nADASYN Summary:')
display(summary_table(adasyn_results).style.highlight_max(color='lightgreen'))

## 7. GAN — Generative Adversarial Network

A GAN consists of two neural networks trained adversarially:
- **Generator**: learns to produce synthetic fraud samples from random noise
- **Discriminator**: learns to distinguish real from synthetic fraud samples

We train the GAN only on fraud samples, then use it to generate additional fraudulent transactions.

In [ ]:
# Extract fraud samples from training set for GAN training
X_fraud_train = X_train[y_train == 1].values
n_features    = X_fraud_train.shape[1]
latent_dim    = 32

print(f'Fraud samples for GAN training : {X_fraud_train.shape[0]}')
print(f'Feature dimensions             : {n_features}')
print(f'GAN latent space dimension     : {latent_dim}')

In [ ]:
def build_generator(latent_dim, n_features):
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(latent_dim,)),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(n_features, activation='tanh'),  # tanh for normalized data
    ], name='Generator')
    return model


def build_discriminator(n_features):
    model = keras.Sequential([
        layers.Dense(256, activation='leaky_relu', input_shape=(n_features,)),
        layers.Dropout(0.3),
        layers.Dense(128, activation='leaky_relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='leaky_relu'),
        layers.Dense(1, activation='sigmoid'),
    ], name='Discriminator')
    return model


generator     = build_generator(latent_dim, n_features)
discriminator = build_discriminator(n_features)

generator.summary()
discriminator.summary()

In [ ]:
# Compile discriminator
discriminator.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Build GAN (generator + frozen discriminator)
discriminator.trainable = False
gan_input  = keras.Input(shape=(latent_dim,))
gan_output = discriminator(generator(gan_input))
gan        = keras.Model(gan_input, gan_output, name='GAN')
gan.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss='binary_crossentropy'
)

print('GAN compiled successfully.')

In [ ]:
# ─── GAN Training ────────────────────────────────────────────────────────────
EPOCHS     = 2000
BATCH_SIZE = 64
n_samples  = len(X_fraud_train)

d_losses, g_losses = [], []

real_labels = np.ones((BATCH_SIZE, 1))   # real = 1
fake_labels = np.zeros((BATCH_SIZE, 1))  # fake = 0

# Label smoothing reduces overconfidence in the discriminator
real_labels_smooth = np.ones((BATCH_SIZE, 1)) * 0.9

for epoch in range(EPOCHS):
    # ── Train Discriminator ──
    idx        = np.random.randint(0, n_samples, BATCH_SIZE)
    real_batch = X_fraud_train[idx]

    noise      = np.random.normal(0, 1, (BATCH_SIZE, latent_dim))
    fake_batch = generator.predict(noise, verbose=0)

    discriminator.trainable = True
    d_loss_real = discriminator.train_on_batch(real_batch, real_labels_smooth)
    d_loss_fake = discriminator.train_on_batch(fake_batch, fake_labels)
    d_loss = 0.5 * (d_loss_real[0] + d_loss_fake[0])

    # ── Train Generator ──
    noise  = np.random.normal(0, 1, (BATCH_SIZE, latent_dim))
    discriminator.trainable = False
    g_loss = gan.train_on_batch(noise, real_labels)

    d_losses.append(d_loss)
    g_losses.append(g_loss)

    if (epoch + 1) % 500 == 0:
        print(f'Epoch {epoch+1:5d}/{EPOCHS}  |  D loss: {d_loss:.4f}  G loss: {g_loss:.4f}')

print('\nGAN training complete.')

In [ ]:
# Plot GAN training curves
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(d_losses, label='Discriminator Loss', color='#2196F3', alpha=0.7)
ax.plot(g_losses, label='Generator Loss',     color='#F44336', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('GAN Training Losses', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gan_training.png', bbox_inches='tight')
plt.show()

In [ ]:
# Generate synthetic fraud samples — enough to balance the training set
n_genuine_train = (y_train == 0).sum()
n_fraud_train   = (y_train == 1).sum()
n_synthetic     = n_genuine_train - n_fraud_train

noise           = np.random.normal(0, 1, (n_synthetic, latent_dim))
synthetic_fraud = generator.predict(noise, verbose=0)

# Combine with original training data
X_train_gan = np.vstack([X_train.values, synthetic_fraud])
y_train_gan = np.hstack([y_train.values, np.ones(n_synthetic)])

# Shuffle
shuffle_idx = np.random.permutation(len(X_train_gan))
X_train_gan = X_train_gan[shuffle_idx]
y_train_gan = y_train_gan[shuffle_idx]

print(f'Original training set  : {len(X_train):,} samples  |  Fraud: {y_train.sum():,} ({y_train.mean()*100:.3f}%)')
print(f'GAN-augmented training : {len(X_train_gan):,} samples  |  Fraud: {int(y_train_gan.sum()):,} ({y_train_gan.mean()*100:.3f}%)')

In [ ]:
# Validate synthetic data quality — compare distributions of a key feature
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat_idx, feat_name in zip(axes, [0, 3, 9], ['V1', 'V4', 'V10']):
    ax.hist(X_fraud_train[:, feat_idx],    bins=50, alpha=0.6, color='#F44336', label='Real Fraud',      density=True)
    ax.hist(synthetic_fraud[:, feat_idx],  bins=50, alpha=0.6, color='#9C27B0', label='Synthetic (GAN)', density=True)
    ax.set_title(f'{feat_name} Distribution', fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Real vs GAN-Synthetic Fraud — Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gan_quality.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Training models on GAN-augmented data...')
gan_results = evaluate_models(X_train_gan, y_train_gan, X_test.values, y_test.values, label='GAN')

print('\nGAN Summary:')
display(summary_table(gan_results).style.highlight_max(color='lightgreen'))

## 8. Full Comparison: Baseline vs SMOTE vs ADASYN vs GAN

In [ ]:
all_results = {
    'Baseline' : baseline_results,
    'SMOTE'    : smote_results,
    'ADASYN'   : adasyn_results,
    'GAN'      : gan_results,
}

# Build a flat comparison table (best model per technique)
comparison_rows = []
for technique, results in all_results.items():
    best_name = max(results, key=lambda k: results[k]['f1'])
    best = results[best_name]
    comparison_rows.append({
        'Technique'    : technique,
        'Best Model'   : best_name,
        'Precision'    : round(best['precision'], 4),
        'Recall'       : round(best['recall'], 4),
        'F1 Score'     : round(best['f1'], 4),
        'ROC-AUC'      : round(best['roc_auc'], 4),
        'Avg Precision': round(best['avg_prec'], 4),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('Technique')
print('Best model per augmentation technique:')
display(comparison_df.style.highlight_max(
    subset=['Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'Avg Precision'],
    color='lightgreen'
))

In [ ]:
# Full comparison per model per technique — F1 heatmap
model_names = list(get_models().keys())
technique_names = list(all_results.keys())

f1_matrix = pd.DataFrame(
    [[all_results[tech][model]['f1'] for model in model_names] for tech in technique_names],
    index=technique_names,
    columns=model_names
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    f1_matrix, annot=True, fmt='.4f', cmap='YlGn',
    linewidths=0.5, ax=ax, vmin=0, vmax=1
)
ax.set_title('F1 Score by Model × Augmentation Technique', fontsize=13, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Technique')
plt.tight_layout()
plt.savefig('f1_comparison_heatmap.png', bbox_inches='tight')
plt.show()

## 9. ROC Curve Comparison (Best Model per Technique)

In [ ]:
colors_tech = ['#607D8B', '#2196F3', '#4CAF50', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for (tech, results), color in zip(all_results.items(), colors_tech):
    best_name = max(results, key=lambda k: results[k]['roc_auc'])
    best = results[best_name]

    fpr, tpr, _ = roc_curve(y_test, best['y_pred_prob'])
    axes[0].plot(fpr, tpr, label=f"{tech} / {best_name} (AUC={best['roc_auc']:.4f})", color=color, lw=2)

    prec, rec, _ = precision_recall_curve(y_test, best['y_pred_prob'])
    axes[1].plot(rec, prec, label=f"{tech} / {best_name} (AP={best['avg_prec']:.4f})", color=color, lw=2)

axes[0].plot([0,1],[0,1],'k--',lw=1.5, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Best Model per Technique', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

baseline_rate = y_test.mean()
axes[1].axhline(y=baseline_rate, color='k', linestyle='--', lw=1.5, label=f'Random (baseline={baseline_rate:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves — Best Model per Technique', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('augmentation_comparison_curves.png', bbox_inches='tight')
plt.show()

## 10. Bar Chart — Metric Comparison Across Techniques

In [ ]:
metrics_to_plot = ['Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
bar_width = 0.18

fig, ax = plt.subplots(figsize=(13, 6))

for i, (row, color) in enumerate(zip(comparison_rows, colors_tech)):
    values = [row[m] for m in metrics_to_plot]
    ax.bar(x + i * bar_width, values, bar_width,
           label=f"{row['Technique']} / {row['Best Model']}",
           color=color, edgecolor='white')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Best Model Performance: Baseline vs Augmentation Techniques', fontsize=13, fontweight='bold')
ax.set_xticks(x + bar_width * 1.5)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('final_comparison.png', bbox_inches='tight')
plt.show()

## 11. Bias Check on Synthetic Data

In [ ]:
# Check that synthetic samples (SMOTE) don't shift the feature distribution dramatically
synthetic_fraud_smote = X_train_smote[len(X_train):]
real_fraud_train      = X_train[y_train == 1]

if len(synthetic_fraud_smote) > 0:
    real_means = real_fraud_train.mean()
    synth_means_smote = pd.DataFrame(synthetic_fraud_smote, columns=X_train.columns).mean()

    drift = (synth_means_smote - real_means).abs().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(14, 5))
    drift.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('SMOTE Bias Check — Mean Drift (Synthetic vs Real Fraud)', fontweight='bold')
    ax.set_ylabel('|Mean Synthetic − Mean Real|')
    ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('bias_check.png', bbox_inches='tight')
    plt.show()

    print(f'Max mean drift: {drift.max():.4f} on feature {drift.idxmax()}')
    print(f'Mean drift across all features: {drift.mean():.4f}')

## 12. Summary & Conclusions

In [ ]:
print('=' * 60)
print('  DATA AUGMENTATION — FINAL CONCLUSIONS')
print('=' * 60)
print()

for row in comparison_rows:
    print(f"  [{row['Technique']:8s}] Best: {row['Best Model']:22s}  F1={row['F1 Score']:.4f}  AUC={row['ROC-AUC']:.4f}")

print()
best_overall = max(comparison_rows, key=lambda r: r['F1 Score'])
print(f"  >>> WINNER: {best_overall['Technique']} / {best_overall['Best Model']}")
print(f"      F1 Score : {best_overall['F1 Score']}")
print(f"      ROC-AUC  : {best_overall['ROC-AUC']}")
print()
print('  Key takeaways:')
print('  - SMOTE/ADASYN improve recall at the cost of some precision')
print('  - GAN quality depends heavily on training epochs and architecture')
print('  - Augmentation is most valuable when recall (catching fraud) matters most')
print('  - Always validate synthetic data quality before deploying')
print('=' * 60)